# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [24]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [43]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf")
docs = loader.load()

In [44]:
document_text = "" 
for page in docs: 
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
# Set summary tone
SUMMARY_TONE = "Victorian English"

In [62]:
# Create Model Class
from pydantic import BaseModel, Field

class ArticleSummary(BaseModel):
    """
    Structured summary of an AI document. Extract Author and Title verbatim. 
    Write Relevance (one paragraph) and Summary (max 1000 tokens) exclusively 
    in the specified tone. Tone field must equal the specified tone. Token counts must come 
    from the API response usage object, not be estimated.
    """
    Author: str = Field(description="Full name(s) of the author(s) of the document")
    Title: str = Field(description="Full title of the document")
    Relevance: str = Field(description="A single paragraph explaining why this document is relevant for an AI professional's development. Write in the specified tone.")
    Summary: str = Field(description="A concise summary of the document, no longer than 1000 tokens. Write in the specified tone.")
    Tone: str = Field(description=f"The exact tone used to write the Summary and Relevance fields. Should be: '{SUMMARY_TONE}'")
    InputTokens: int = Field(description="Number of input/prompt tokens used, taken directly from the API response usage object")
    OutputTokens: int = Field(description="Number of output/completion tokens used, taken directly from the API response usage object")


In [63]:
system_prompt = f"""
You are an expert AI research analyst. 
Your task is to analyze documents and produce structured summaries. 
You must write your Summary and Relevance fields in the following tone: {SUMMARY_TONE}. Be consistent and distinguishable in applying this tone throughout. 
Keep the Summary under 1000 tokens and Relevance to one paragraph.
"""

In [64]:
user_prompt = f"""
Please analyze the following document and extract the required information.

DOCUMENT:
{document_text}
"""

In [65]:
# Initialize the openAI client
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [66]:
completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",  # Not GPT-5 family
    messages=[
        {"role": "developer", "content": system_prompt},
        {"role": "user",      "content": user_prompt},
    ],
    response_format=ArticleSummary,
)

In [67]:
raw = completion.choices[0].message.parsed
result = ArticleSummary(
    Author=raw.Author,
    Title=raw.Title,
    Relevance=raw.Relevance,
    Summary=raw.Summary,
    Tone=raw.Tone,
    InputTokens=completion.usage.prompt_tokens,
    OutputTokens=completion.usage.completion_tokens,
)

In [68]:
print(result.model_dump_json(indent=2))

{
  "Author": "MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This document holds great significance for those within the realm of artificial intelligence, as it elucidates the stark realities faced by organizations amidst the integration of generative AI. The findings detail not only the disparate outcomes of AI adoption but also the underlying causes of such discrepancies, thus serving as a critical guide for practitioners. Revelatory insights are presented on how to navigate the chasms that manifest within the GenAI landscape, offering foresight that may aid in the development of more transformative AI strategies.",
  "Summary": "In the year of our Lord twenty twenty-five, this document titled \"The GenAI Divide\" ventured into the intricate realm of generative artificial intelligence within the business sphere. It was crafted by the venerable minds of MIT NANDA alongside estee

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
